# 7 Wonders — Naprawiony trening

## Co zostało naprawione

### Problem 1: Normalizacja
Stary kod normalizował wektory stanów w Pythonie (obliczał średnią i odchylenie dla całego datasetu),
ale C# podczas gry podawało surowe wartości do modelu ONNX. Model uczył się na innych liczbach
niż dostawał w czasie rzeczywistym.

**Naprawa**: Usunięto normalizację z datasetu. Sieć ma wbudowany `LayerNorm` jako pierwszą warstwę —
normalizuje każdy wektor stanu sama, bez zewnętrznych parametrów.

### Problem 2: Value target
Każdy ruch w grze dostawał +1 lub -1 zależnie od wyniku końcowego.
Ruch w turze 3 z 50-turnowej gry dostawał tę samą etykietę co ruch decydujący.

**Naprawa**: Discount factor `gamma=0.98`. Ruchy bliżej końca mają silniejszy sygnał.
Ostatni ruch: ±1.0, 10 ruchów wcześniej: ±0.82, 25 ruchów wcześniej: ±0.60.

### Problem 3: Rozmiar modelu vs dane
Sieć 1903→512→256 (~1.5M parametrów) na 5700 próbkach overfittowała od epoki 2.

**Naprawa**: Mniejszy backbone 1903→256→128 (~400k parametrów). Dopasowany do ilości danych.

In [1]:
import os
import sys
from pathlib import Path

# --- Konfiguracja ---
EPOCHS      = int(os.getenv('WONDERS_EPOCHS', '100'))
GAMES_TRAIN = int(os.getenv('WONDERS_GAMES', '100'))
SEED        = int(os.getenv('WONDERS_SEED', '1'))
OUTPUT_NAME = os.getenv('WONDERS_OUTPUT_NAME', 'self_play_puct')
BATCH_SIZE  = int(os.getenv('WONDERS_BATCH_SIZE', '64'))   # zwiększono z 32
LR          = float(os.getenv('WONDERS_LR', '3e-4'))       # zmniejszono z 1e-3
PATIENCE    = int(os.getenv('WONDERS_PATIENCE', '15'))
GAMMA       = float(os.getenv('WONDERS_GAMMA', '0.98'))     # discount factor
VALUE_WEIGHT = float(os.getenv('WONDERS_VALUE_WEIGHT', '0.5'))

repo_root    = Path(r'c:/Users/kubeu/Kuba-dokumenty/Magisterka/7 Wonders')
encoding_dir = repo_root / 'GameAI' / 'Encoding'
results_dir  = repo_root / 'GameConsole' / 'Results'
models_dir   = encoding_dir / 'onnx_models'
models_dir.mkdir(exist_ok=True)

sys.path.insert(0, str(encoding_dir))

import importlib
import game_training_pipeline_fixed as gtp
gtp = importlib.reload(gtp)

print(f'State vector size : {gtp.ActionSpace.STATE_VECTOR_SIZE}')
print(f'Action space size : {gtp.ActionSpace.TOTAL_PRIMARY_ACTIONS}')
print(f'LR={LR}, batch={BATCH_SIZE}, epochs={EPOCHS}, gamma={GAMMA}')

State vector size : 1903
Action space size : 120
LR=0.0003, batch=64, epochs=100, gamma=0.98


In [2]:
import torch
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')
if device.type == 'cuda':
    print(f'Karta: {torch.cuda.get_device_name(0)}')

Device: cuda
Karta: NVIDIA GeForce RTX 4050 Laptop GPU


## Wczytanie danych

Notebook szuka najnowszego pliku JSON w katalogu Results.
Jeśli C# zacznie eksportować pliki `.bin` (format binarny), zmień `GameDataset` na `BinaryGameDataset`.

In [3]:
import random
import copy
from torch.utils.data import DataLoader, Subset

# Znajdź najnowszy plik danych
candidates = sorted(results_dir.glob(f'{OUTPUT_NAME}_*_games*.json'))
candidates += sorted(results_dir.glob('training_*.json'))
candidates = [p for p in candidates if p.exists()]
if not candidates:
    raise FileNotFoundError(f'Brak plików JSON w {results_dir}')

latest_file = max(candidates, key=lambda p: p.stat().st_mtime)
print(f'Używam pliku: {latest_file}')

# Wczytaj dataset
# gamma=0.98 to discount factor — patrz komentarze w game_training_pipeline_fixed.py
dataset = gtp.GameDataset(str(latest_file), gamma=GAMMA, validate_shapes=True)

# Podział train/val PO MECZACH, nie po ruchach
# To ważne: jeśli dzielimy po ruchach, ruchy z tego samego meczu trafiają
# zarówno do train jak i val — model "zna odpowiedź" i val_loss jest zawyżony.
indices_by_match = {}
for i, sample in enumerate(dataset.data):
    mid = sample.get('match_id', f'fallback_{i}')
    indices_by_match.setdefault(mid, []).append(i)

match_ids = list(indices_by_match.keys())
rng = random.Random(42)
rng.shuffle(match_ids)

n_train = max(1, int(len(match_ids) * 0.8))  # 80/20 zamiast 60/40
train_ids = match_ids[:n_train]
val_ids   = match_ids[n_train:]

train_idx = [i for mid in train_ids for i in indices_by_match[mid]]
val_idx   = [i for mid in val_ids   for i in indices_by_match[mid]]

print(f'Mecze: train={len(train_ids)}, val={len(val_ids)}')
print(f'Próbki: train={len(train_idx)}, val={len(val_idx)}')

train_loader = DataLoader(Subset(dataset, train_idx), batch_size=BATCH_SIZE, shuffle=True)
val_loader   = DataLoader(Subset(dataset, val_idx),   batch_size=BATCH_SIZE, shuffle=False) if val_idx else None

Używam pliku: c:\Users\kubeu\Kuba-dokumenty\Magisterka\7 Wonders\GameConsole\Results\self_play_puct_20260518_144032_100_games_minimal.json


2026-05-19 11:21:39,643 - INFO - Wczytano 5772 próbek


Mecze: train=80, val=20
Próbki: train=4602, val=1170


## Budowanie modelu

Nowa sieć `PolicyNetwork` ma:
- `LayerNorm` na wejściu zamiast zewnętrznej normalizacji
- Mniejszy backbone (dopasowany do ilości danych)
- `Tanh` na wyjściu value head (wymusza zakres [-1, +1])

In [4]:
model = gtp.PolicyNetwork(
    state_dim=gtp.ActionSpace.STATE_VECTOR_SIZE,
    hidden_dim=128,
    dropout=0.1,
).to(device)

n_params = sum(p.numel() for p in model.parameters())
print(f'Parametry modelu: {n_params:,}')
print(f'Próbki treningowe: {len(train_idx)}')
print(f'Stosunek próbki/parametry: {len(train_idx)/n_params:.4f}')
print('(Chcemy >0.01; im wyższy tym mniej overfittingu)')

# Szybki test — czy model w ogóle działa
with torch.no_grad():
    test_batch = next(iter(train_loader))
    out = model(test_batch['state'].to(device), test_batch['action_mask'].to(device))
    print(f"\nTest forward pass OK:")
    print(f"  policy_masked_logits: {out['policy_masked_logits'].shape}")
    print(f"  value: {out['value'].shape}")
    print(f"  value zakres: [{out['value'].min().item():.3f}, {out['value'].max().item():.3f}]")

Parametry modelu: 564,439
Próbki treningowe: 4602
Stosunek próbki/parametry: 0.0082
(Chcemy >0.01; im wyższy tym mniej overfittingu)

Test forward pass OK:
  policy_masked_logits: torch.Size([64, 120])
  value: torch.Size([64, 1])
  value zakres: [-0.145, -0.011]


## Trening

In [5]:
optimizer = torch.optim.Adam(
    model.parameters(),
    lr=LR,
    weight_decay=1e-4,  # L2 regularyzacja — dodatkowa ochrona przed overfittingiem
)

# CosineAnnealingLR: learning rate płynnie spada od LR do ~0 przez EPOCHS epok
# To lepsze niż stały LR — na początku eksplorujemy (duży krok), potem finiszujemy (mały krok)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS, eta_min=LR/20)

if val_loader:
    baseline = gtp.evaluate(model, val_loader, device, VALUE_WEIGHT)
    print(f'Val loss przed treningiem: {baseline["total_loss"]:.4f}')

best_val_loss  = float('inf')
best_epoch     = 0
best_state     = None
no_improve     = 0

for epoch in range(EPOCHS):
    train_stats = gtp.train_epoch(model, train_loader, optimizer, device, VALUE_WEIGHT)
    scheduler.step()

    if val_loader:
        val_stats = gtp.evaluate(model, val_loader, device, VALUE_WEIGHT)
        current_lr = scheduler.get_last_lr()[0]

        if val_stats['total_loss'] < best_val_loss - 1e-5:
            best_val_loss = val_stats['total_loss']
            best_epoch    = epoch + 1
            best_state    = copy.deepcopy(model.state_dict())
            no_improve    = 0
            marker = ' ← best'
        else:
            no_improve += 1
            marker = ''

        print(
            f'epoch {epoch+1:3d}/{EPOCHS} | '
            f'train {train_stats["total_loss"]:.4f} '
            f'(p={train_stats["policy_loss"]:.3f} v={train_stats["value_loss"]:.3f}) | '
            f'val {val_stats["total_loss"]:.4f} '
            f'(p={val_stats["policy_loss"]:.3f} v={val_stats["value_loss"]:.3f}) | '
            f'lr={current_lr:.1e}{marker}'
        )

        if no_improve >= PATIENCE:
            print(f'Early stopping: brak poprawy przez {PATIENCE} epok')
            break
    else:
        print(f'epoch {epoch+1:3d}/{EPOCHS} | train {train_stats["total_loss"]:.4f}')
        if best_state is None:
            best_state = copy.deepcopy(model.state_dict())
            best_epoch = 1

if best_state is not None:
    model.load_state_dict(best_state)
    print(f'\nPrzywrócono checkpoint z epoki {best_epoch} (val loss: {best_val_loss:.4f})')

Val loss przed treningiem: 1.6992
epoch   1/100 | train 1.5185 (p=1.387 v=0.263) | val 1.7028 (p=1.393 v=0.620) | lr=3.0e-04 ← best
epoch   2/100 | train 1.3937 (p=1.357 v=0.074) | val 1.7642 (p=1.390 v=0.748) | lr=3.0e-04
epoch   3/100 | train 1.3645 (p=1.354 v=0.021) | val 1.7430 (p=1.389 v=0.708) | lr=3.0e-04
epoch   4/100 | train 1.3565 (p=1.351 v=0.011) | val 1.7385 (p=1.389 v=0.700) | lr=3.0e-04
epoch   5/100 | train 1.3527 (p=1.349 v=0.008) | val 1.7312 (p=1.388 v=0.686) | lr=3.0e-04
epoch   6/100 | train 1.3493 (p=1.346 v=0.006) | val 1.7312 (p=1.388 v=0.687) | lr=3.0e-04
epoch   7/100 | train 1.3461 (p=1.343 v=0.005) | val 1.7316 (p=1.387 v=0.689) | lr=3.0e-04
epoch   8/100 | train 1.3434 (p=1.341 v=0.005) | val 1.7360 (p=1.388 v=0.696) | lr=3.0e-04
epoch   9/100 | train 1.3407 (p=1.338 v=0.004) | val 1.7265 (p=1.388 v=0.677) | lr=2.9e-04
epoch  10/100 | train 1.3380 (p=1.336 v=0.004) | val 1.7311 (p=1.388 v=0.687) | lr=2.9e-04
epoch  11/100 | train 1.3354 (p=1.333 v=0.004) | 

## Eksport do ONNX

Model jest eksportowany do ONNX z wbudowanym `LayerNorm`.
C# nie musi nic zmieniać — nadal podaje surowe wartości stanu, a model normalizuje je sam.

In [6]:
import re
from datetime import datetime

timestamp = datetime.now().strftime('%Y%m%d_%H%M')
onnx_name = f'policy_fixed_{timestamp}_ep{best_epoch}.onnx'
onnx_path = models_dir / onnx_name

model.onnx_export(str(onnx_path), validate=True)
print(f'Model zapisany: {onnx_path}')

# Weryfikacja: czy model w ONNX daje te same wyniki co PyTorch?
try:
    import onnxruntime as ort
    sess = ort.InferenceSession(str(onnx_path))

    sample = next(iter(val_loader or train_loader))
    state_np = sample['state'][:1].numpy()
    mask_np  = sample['action_mask'][:1].numpy()

    onnx_out = sess.run(None, {'state': state_np, 'action_mask': mask_np})
    torch_out = model(
        sample['state'][:1].to(device),
        sample['action_mask'][:1].to(device)
    )

    diff_policy = abs(onnx_out[0] - torch_out['policy_masked_logits'].cpu().detach().numpy()).max()
    diff_value  = abs(onnx_out[1] - torch_out['value'].cpu().detach().numpy()).max()
    print(f'Różnica PyTorch vs ONNX: policy={diff_policy:.6f}, value={diff_value:.6f}')
    print('(Powinno być <0.001 — to błąd numeryczny, nie błąd logiczny)')
except ImportError:
    print('onnxruntime nie zainstalowane — pomijam weryfikację')
    print('Możesz zainstalować: pip install onnxruntime')

2026-05-19 11:21:51,262 - INFO - Eksportuję model do: c:\Users\kubeu\Kuba-dokumenty\Magisterka\7 Wonders\GameAI\Encoding\onnx_models\policy_fixed_20260519_1121_ep1.onnx
2026-05-19 11:21:51,402 - INFO - ✓ ONNX OK: c:\Users\kubeu\Kuba-dokumenty\Magisterka\7 Wonders\GameAI\Encoding\onnx_models\policy_fixed_20260519_1121_ep1.onnx


Model zapisany: c:\Users\kubeu\Kuba-dokumenty\Magisterka\7 Wonders\GameAI\Encoding\onnx_models\policy_fixed_20260519_1121_ep1.onnx
onnxruntime nie zainstalowane — pomijam weryfikację
Możesz zainstalować: pip install onnxruntime
